In [4]:
import json
import os
from openai import OpenAI

In [5]:
# Connect directly to your local background Ollama engine
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # Ollama ignores api-keys, but a string wrapper is required
)

In [40]:
MODEL_NAME = "huihui_ai/nemotron-v1-abliterated:8b-llama-3.1-nano"
PROMPT_FILE = "prompt/conversation-generator.md"
# 2. Read your custom prompt.md file safely
if not os.path.exists(PROMPT_FILE):
    raise FileNotFoundError(f"Could not find {PROMPT_FILE}! Ensure it is in the same folder as this script.")

with open(PROMPT_FILE, "r", encoding="utf-8") as f:
    system_instructions = f.read()

# 3. Define the list of inputs/domains you want the model to generate datasets for
# (You can modify this list or read it from a separate text file)
target_domains = [
    "Home Appliances",

]

generated_dataset = []

print("🚀 Starting dataset generation using prompt.md rules...")

# 4. Run the generation loop
for index, domain in enumerate(target_domains, 1):
    print(f"[{index}/{len(target_domains)}] Generating data for: {domain}...")
    
    # Construct the user message telling Nemotron what specific topic to apply to prompt.md
    user_message = f"Apply your system guidelines to generate a high-quality, diverse dataset sample for the following domain: {domain}"
    
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_instructions}, # Injects your prompt.md rules
                # {"role": "user", "content": user_message}
            ],
            temperature=0.95, # Balanced for structural adherence and creative diversity
            max_tokens=2500  # Ensure enough runway for complex schemas
        )
        
        choice = response.choices[0]
        if hasattr(choice, 'message'):
            raw_output = choice.message.content.strip()
        else:
            raw_output = choice['message']['content'].strip()

        
        # Save each raw generation block into our tracking list
        generated_dataset.append({
            "id": index,
            "domain": domain,
            "generated_output": raw_output
        })
        
    except Exception as e:
        print(f"❌ Error during generation for {domain}: {e}")



🚀 Starting dataset generation using prompt.md rules...
[1/1] Generating data for: Home Appliances...


In [41]:
# 5. Save the final raw synthetic blocks to a JSON file
output_file = "synthetic_dataset_output.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(generated_dataset, f, indent=4, ensure_ascii=False)

print(f"\n✅ Done! Your generated dataset has been saved to: {output_file}")


✅ Done! Your generated dataset has been saved to: synthetic_dataset_output.json


In [42]:
import json
import re

# 1. Load the raw, messy dataset generated by Nemotron
input_filename = "synthetic_dataset_output.json"
output_filename = "final_training_data.jsonl"

with open(input_filename, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

cleaned_count = 0

# 2. Open the clean .jsonl output file
with open(output_filename, "w", encoding="utf-8") as f_out:
    for item in raw_data:
        text_content = item.get("generated_output", "")
        print(text_content)  # Debug: See the raw output before cleaning
        # Use regex to find everything between curly brackets { ... }
        # This extracts only the raw JSON data blocks and ignores text chatter
        json_blocks = re.findall(r"\{.*?\}", text_content, re.DOTALL)
        
        for block in json_blocks:
            try:
                # Validate that it's legitimate JSON data
                parsed_json = json.loads(block.strip())
                
                # Filter out small irrelevant fragments that might misalign
                if "question" in parsed_json or "answer" in parsed_json:
                    # Write as a single compressed line (JSONL format requirement)
                    f_out.write(json.dumps(parsed_json, ensure_ascii=False) + "\n")
                    cleaned_count += 1
            except json.JSONDecodeError:
                # Skips any fragments that are not valid JSON objects
                continue

print(f"🎉 Done! Extracted {cleaned_count} clean sample objects into: '{output_filename}'")


think>

```json
{"question": "[{\"role\": \"user\", \"content\": \"{\"I'm feeling exhausted and want to\\nwind everything down for the night, can you assist?\"}\\n},{\"role\": \"assistant\", \"content\": \"\", \"{tool_calls\": [{type: \\\"function\\\", function: {name: \\\"set_scene\\>, arguments: {scene: \\\"bedtime\\\"}}}]},\",\"role\": \"user\", \"content\": \"{\"It's getting too cold in here, can you warm it up?\\n},{\"role\": \"assistant\", \"content\": \"\", \"{tool_calls\": [{type: \\\"function\\>, function: {name: \\\"set_thermostat\\>, arguments: {temperature: 70, mode: \\\"heat\\\"}}}]},\",\"role\": \"user\", \"content\": \"{\"Please lock the front door as I'm about to head home for\\nbedtime.\\n},{\"role\": \"assistant\", \"content\": \"\", \"{tool_calls\": [{type: \\\"function\\>, function: {name: \\\"lock_door\\>, arguments: {door: \\\"front\\\", state: \\\"lock\\\"}}}]},\",\"role\": \"user\", \"content\": \"{\\\"I'm sorry, I couldn't understand that. Could you please\\nre